# LeetCode 528 — Random Pick with Weight bằng Python

Link đề: https://leetcode.com/problems/random-pick-with-weight/ · Độ khó: **Medium** · Chủ đề: Array, Math, Binary Search, Prefix Sum, Randomized, Design

Skeleton LeetCode:

```python
class Solution:

    def __init__(self, w: List[int]):
        ...

    def pickIndex(self) -> int:
        ...


# Your Solution object will be instantiated and called as such:
# obj = Solution(w)
# param_1 = obj.pickIndex()
```

Ghi chú: đây là bài **thiết kế lớp + ngẫu nhiên**. Kết quả mỗi lần gọi `pickIndex()` khác nhau, nên ta **không so đáp án cứng**. Notebook này kiểm tra bằng hai cách mạnh hơn: (1) **đếm chính xác bằng `Fraction`** — duyệt *mọi* chuỗi lựa chọn ngẫu nhiên có thể xảy ra rồi so với `w[i] / sum(w)`; (2) **test thống kê chi-bình-phương** với RNG có seed cố định.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: chọn một chỉ số `i` với xác suất **đúng bằng** `w[i] / sum(w)`, và gọi được rất nhiều lần (tới `10^4` lần) nên mỗi lần gọi phải nhanh.
2. Nắm các công cụ Python: `random.randint`, `random.seed`, tổng tiền tố `itertools.accumulate`, tìm kiếm nhị phân `bisect`, phân số chính xác `fractions.Fraction`, và cách **tự dựng bộ sinh số ngẫu nhiên có seed** (xorshift) để kết quả chạy lại giống hệt.
3. Đi từ **cách ngây thơ** (nhân bản chỉ số `w[i]` lần) → **tổng tiền tố + quét tuyến tính** → **tổng tiền tố + tìm kiếm nhị phân** → **phương pháp alias (Walker/Vose)** chọn trong `O(1)`.
4. Hiểu mấu chốt: biến "xác suất theo trọng số" thành "**chọn đều một số nguyên trong `[1, S]`**, rồi hỏi số đó rơi vào đoạn của chỉ số nào" — mỗi chỉ số `i` sở hữu đúng `w[i]` số nguyên.
5. Biết **test code ngẫu nhiên đúng cách**: đếm chính xác bằng `Fraction` bằng cách liệt kê toàn bộ khả năng, kiểm định chi-bình-phương có seed, và các lỗi kinh điển (lệch 1 ở biên, `bisect_right`, bỏ qua trọng số, dựng lại mảng mỗi lần gọi).

## 2. Đề bài & ví dụ

Cho mảng số nguyên **dương** `w` (0-indexed), trong đó `w[i]` là **trọng số** của chỉ số `i`.

Hãy cài lớp `Solution`:

- `Solution(w)`: khởi tạo đối tượng với mảng trọng số `w`.
- `pickIndex()`: chọn **ngẫu nhiên** một chỉ số trong `[0, len(w) - 1]` và trả về. **Xác suất chọn chỉ số `i` là `w[i] / sum(w)`.**

Ví dụ: `w = [1, 3]` thì xác suất chọn chỉ số `0` là `1 / 4 = 25%`, chỉ số `1` là `3 / 4 = 75%`.

**Ví dụ 1:**

```
Input
["Solution", "pickIndex"]
[[[1]], []]
Output
[null, 0]
```

Chỉ có một phần tử nên luôn trả `0`.

**Ví dụ 2:**

```
Input
["Solution", "pickIndex", "pickIndex", "pickIndex", "pickIndex", "pickIndex"]
[[[1, 3]], [], [], [], [], []]
Output
[null, 1, 1, 1, 1, 0]
```

Vì là bài ngẫu nhiên nên **nhiều đáp án đều đúng**: `[null,1,1,1,1,1]`, `[null,1,1,1,0,0]`, ... đều hợp lệ. Điều được chấm là *phân phối xác suất*, không phải một dãy cụ thể.

**Ràng buộc:**

- `1 <= w.length <= 10^4`
- `1 <= w[i] <= 10^5`
- `pickIndex` được gọi tối đa `10^4` lần.

Chú ý: tổng `S = sum(w)` có thể tới `10^4 × 10^5 = 10^9`. Con số này sẽ loại ngay cách làm "nhân bản phần tử".

In [1]:
import bisect
import contextlib
import math
import random
import sys
import time
from collections import Counter
from fractions import Fraction
from itertools import accumulate
from typing import List


def run_ops(cls, names, args):
    """Mô phỏng cách LeetCode chạy: dòng 1 là lệnh, dòng 2 là tham số.
    names[0] == "Solution" nghĩa là gọi constructor; các lệnh sau gọi method cùng tên."""
    obj, out = None, []
    for name, a in zip(names, args):
        if name == "Solution":
            obj = cls(*a)
            out.append(None)
        else:
            out.append(getattr(obj, name)(*a))
    return out


class DapAnMau:
    """Lớp giả chỉ để minh hoạ bộ chạy: trả lại đúng dãy của ví dụ 2 (một đáp án hợp lệ)."""
    def __init__(self, w):
        self.kq = [1, 1, 1, 1, 0]
    def pickIndex(self):
        return self.kq.pop(0)


names = ["Solution", "pickIndex", "pickIndex", "pickIndex", "pickIndex", "pickIndex"]
args = [[[1, 3]], [], [], [], [], []]
print(run_ops(DapAnMau, names, args))   # [None, 1, 1, 1, 1, 0] (None tương ứng "null")

[None, 1, 1, 1, 1, 0]


## 3. Kiến thức nền

Bài này dùng 6 công cụ nhỏ. Mỗi công cụ ta thử riêng trước khi ghép lại.

### 3.1 Số ngẫu nhiên và `seed`

`random.randint(a, b)` trả về một số nguyên **ngẫu nhiên đều** trong đoạn `[a, b]` (**cả hai đầu đều lấy được**). "Đều" nghĩa là mọi số trong đoạn có xác suất như nhau.

Máy tính không có ngẫu nhiên thật mà dùng **bộ sinh giả ngẫu nhiên** (PRNG): một công thức biến *trạng thái* thành số kế tiếp. Trạng thái đầu gọi là **seed**. Cùng seed → cùng dãy số → kết quả chạy lại giống hệt. Đây là chìa khoá để test code ngẫu nhiên: ta cố định seed để notebook luôn cho cùng output.

In [2]:
random.seed(7)
a = [random.randint(1, 6) for _ in range(8)]
random.seed(7)                       # đặt lại seed -> dãy lặp lại y hệt
b = [random.randint(1, 6) for _ in range(8)]
print(a)
print(b)
print("Giống nhau:", a == b)

# randint lấy được CẢ HAI đầu: thử 2000 lần, xem tập giá trị xuất hiện
random.seed(1)
print(sorted(set(random.randint(1, 4) for _ in range(2000))))

[3, 2, 4, 6, 1, 1, 5, 1]
[3, 2, 4, 6, 1, 1, 5, 1]
Giống nhau: True
[1, 2, 3, 4]


### 3.2 Tự dựng một bộ sinh số có seed (xorshift)

Để hiểu "seed" không phải phép màu, ta tự viết một PRNG nhỏ tên **xorshift64**. Nó chỉ dùng phép dịch bit và XOR:

```
x ^= x << 13
x ^= x >> 7
x ^= x << 17        (luôn giữ 64 bit thấp)
```

Hai chi tiết quan trọng:

- **Trộn seed** bằng `splitmix64` để seed `1, 2, 3...` cho trạng thái đầu khác hẳn nhau (xorshift không chịu trạng thái `0`).
- **Chọn đều trong `[0, n)`** không được viết `x % n` vì có **thiên lệch modulo**. Ví dụ có 256 giá trị `0..255` rồi lấy `% 10`: các chữ số `0..5` xuất hiện 26 lần, còn `6..9` chỉ 25 lần. Cách đúng là **lấy mẫu loại bỏ** (rejection sampling): bỏ các giá trị rơi vào phần "dư" ở cuối rồi mới lấy `%`.

In [3]:
# Thiên lệch modulo: 256 giá trị 0..255 chia cho 10 lấy dư
c = Counter(v % 10 for v in range(256))
print("Số lần mỗi chữ số:", [c[d] for d in range(10)])


class XorShift64:
    """PRNG tự dựng có seed. Dùng làm 'rng' cho các hàm minh hoạ trong notebook."""
    MASK = (1 << 64) - 1

    def __init__(self, seed):
        z = (seed + 0x9E3779B97F4A7C15) & self.MASK          # splitmix64: trộn seed
        z = ((z ^ (z >> 30)) * 0xBF58476D1CE4E5B9) & self.MASK
        z = ((z ^ (z >> 27)) * 0x94D049BB133111EB) & self.MASK
        z ^= z >> 31
        self.state = z or 0x2545F4914F6CDD1D                  # tránh trạng thái 0

    def next_u64(self):
        x = self.state
        x ^= (x << 13) & self.MASK
        x ^= x >> 7
        x ^= (x << 17) & self.MASK
        self.state = x
        return x

    def randrange(self, n):
        """Số nguyên đều trong [0, n): lấy mẫu loại bỏ để KHÔNG lệch modulo."""
        limit = (1 << 64) - ((1 << 64) % n)                  # bội của n lớn nhất <= 2^64
        while True:
            r = self.next_u64()
            if r < limit:
                return r % n

    def randint(self, a, b):
        """Số nguyên đều trong [a, b] (cả hai đầu)."""
        return a + self.randrange(b - a + 1)


r1, r2 = XorShift64(42), XorShift64(42)
print([r1.randint(1, 6) for _ in range(10)])
print([r2.randint(1, 6) for _ in range(10)])   # cùng seed -> cùng dãy
print(XorShift64(43).randint(1, 10 ** 9), "<- seed khác, số khác")

Số lần mỗi chữ số: [26, 26, 26, 26, 26, 26, 25, 25, 25, 25]
[5, 3, 1, 6, 3, 1, 2, 5, 5, 5]
[5, 3, 1, 6, 3, 1, 2, 5, 5, 5]
864427096 <- seed khác, số khác


### 3.3 Tổng tiền tố (prefix sum)

Cho `w = [3, 1, 2, 4]`. **Tổng tiền tố** cộng dồn từ trái sang: `[3, 3+1, 3+1+2, 3+1+2+4] = [3, 4, 6, 10]`. Phần tử cuối là tổng `S = 10`.

Python có `itertools.accumulate` làm đúng việc này. Ý nghĩa hình học: đặt các đoạn thẳng có độ dài `w[i]` nối đuôi nhau; `prefix[i]` là **toạ độ đầu mút phải** của đoạn thứ `i`.

In [4]:
w = [3, 1, 2, 4]
prefix = list(accumulate(w))
print("w      =", w)
print("prefix =", prefix, " tổng S =", prefix[-1])

# cùng việc, viết bằng vòng lặp để thấy rõ
prefix2, running = [], 0
for x in w:
    running += x
    prefix2.append(running)
print("viết tay:", prefix2, prefix2 == prefix)

w      = [3, 1, 2, 4]
prefix = [3, 4, 6, 10]  tổng S = 10
viết tay: [3, 4, 6, 10] True


### 3.4 Tìm kiếm nhị phân và `bisect`

Mảng tiền tố **tăng chặt** (vì `w[i] >= 1`) nên đã **sắp xếp**. Với mảng sắp xếp, ta tìm "vị trí đầu tiên có giá trị `>= x`" bằng **tìm kiếm nhị phân**: mỗi bước nhìn phần tử giữa rồi loại nửa còn lại, nên chỉ mất `O(log n)`.

Thư viện chuẩn có `bisect.bisect_left(a, x)`: trả về chỉ số `i` nhỏ nhất sao cho `a[i] >= x` (nếu mọi phần tử `< x` thì trả `len(a)`). Còn `bisect_right(a, x)` trả chỉ số nhỏ nhất sao cho `a[i] > x`. **Hai hàm khác nhau đúng khi `x` bằng một phần tử** — điểm này sẽ gây lỗi ở mục 13.

In [5]:
p = [3, 4, 6, 10]
for x in [1, 3, 4, 5, 6, 10, 11]:
    print(f"x={x:2d}  bisect_left={bisect.bisect_left(p, x)}  bisect_right={bisect.bisect_right(p, x)}")

x= 1  bisect_left=0  bisect_right=0
x= 3  bisect_left=0  bisect_right=1
x= 4  bisect_left=1  bisect_right=2
x= 5  bisect_left=2  bisect_right=2
x= 6  bisect_left=2  bisect_right=3
x=10  bisect_left=3  bisect_right=4
x=11  bisect_left=4  bisect_right=4


### 3.5 `Fraction`: phân số chính xác

Số thực dấu phẩy động (`0.1 + 0.2 = 0.30000000000000004`) không so sánh bằng `==` được. `fractions.Fraction(a, b)` giữ **phân số đúng**, tự rút gọn, nên ta so được xác suất "bằng nhau tuyệt đối". Chúng ta sẽ dùng nó để chứng minh `P(chọn i) = w[i] / S` mà không phải "chạy nhiều lần rồi hy vọng".

In [6]:
print(0.1 + 0.2 == 0.3)                                 # False (sai số thực)
print(Fraction(1, 10) + Fraction(2, 10) == Fraction(3, 10))   # True (chính xác)
print(Fraction(6, 8), Fraction(3, 12) + Fraction(1, 4))

False
True
3/4 1/2


### 3.6 Kiểm định chi-bình-phương (chi-square)

Làm sao biết một bộ chọn ngẫu nhiên "đúng phân phối" khi mỗi lần chạy ra khác nhau? Chạy `N` lần, đếm số lần `O_i` mỗi chỉ số xuất hiện, so với kỳ vọng `E_i = N · w[i] / S`:

```
chi2 = Σ (O_i − E_i)² / E_i
```

Nếu phân phối đúng, `chi2` xấp xỉ phân phối chi-bình-phương với `k − 1` bậc tự do (`k` là số chỉ số). Ta đổi `chi2` thành **p-value** = xác suất thấy độ lệch lớn hơn hoặc bằng như vậy *nếu code đúng*. Quy ước: p-value rất nhỏ (ví dụ `< 0.001`) → nghi ngờ code sai.

Không có `scipy`, ta tự viết hàm p-value bằng hàm gamma bất toàn (chỉ cần `math.lgamma`). Kiểm chứng hàm này bằng các giá trị tới hạn nổi tiếng: với `df = 1`, `chi2 = 3.841` ứng với p = 0.05; với `df = 3`, `chi2 = 7.815` cũng p = 0.05.

In [7]:
def chi2_sf(x, df):
    """P(chi2 >= x) với df bậc tự do = Q(df/2, x/2) (hàm gamma bất toàn trên đã chuẩn hoá)."""
    a, xx = df / 2, x / 2
    if xx <= 0:
        return 1.0
    if xx < a + 1:                              # chuỗi lũy thừa
        term = s = 1 / a
        n = a
        for _ in range(2000):
            n += 1
            term *= xx / n
            s += term
            if abs(term) < abs(s) * 1e-16:
                break
        return 1 - s * math.exp(-xx + a * math.log(xx) - math.lgamma(a))
    b = xx + 1 - a                              # phân số liên tục (Lentz)
    c, d = 1e300, 1 / b
    h = d
    for i in range(1, 2000):
        an = -i * (i - a)
        b += 2
        d = an * d + b
        d = 1e-300 if abs(d) < 1e-300 else d
        c = b + an / c
        c = 1e-300 if abs(c) < 1e-300 else c
        d = 1 / d
        delta = d * c
        h *= delta
        if abs(delta - 1) < 1e-16:
            break
    return math.exp(-xx + a * math.log(xx) - math.lgamma(a)) * h


def chi_square_test(counts, weights):
    """Trả (thống kê chi2, p-value) của đếm quan sát so với kỳ vọng theo trọng số."""
    N, S = sum(counts), sum(weights)
    stat = 0.0
    for o, wt in zip(counts, weights):
        e = N * wt / S
        stat += (o - e) ** 2 / e
    return stat, chi2_sf(stat, len(weights) - 1)


print(round(chi2_sf(3.841458820694124, 1), 6), round(chi2_sf(7.814727903251179, 3), 6),
      round(chi2_sf(18.307038053275146, 10), 6))   # đều ra 0.05

# Thử trên xúc xắc 6 mặt từ XorShift64 (đúng) và xúc xắc bị lệch (sai)
rng = XorShift64(2024)
fair = Counter(rng.randint(1, 6) for _ in range(60000))
counts = [fair[i] for i in range(1, 7)]
print("xúc xắc đều :", counts, "-> chi2, p =", [round(v, 4) for v in chi_square_test(counts, [1] * 6)])
loaded = [10200, 10200, 9800, 9800, 10000, 10000]
print("xúc xắc lệch:", loaded, "-> chi2, p =", [round(v, 6) for v in chi_square_test(loaded, [1] * 6)])

0.05 0.05 0.05
xúc xắc đều : [10096, 10190, 9878, 9788, 10032, 10016] -> chi2, p = [10.6424, 0.0589]
xúc xắc lệch: [10200, 10200, 9800, 9800, 10000, 10000] -> chi2, p = [16.0, 0.006844]


## 4. Ý tưởng nền tảng / trực giác

Hãy tưởng tượng một **thanh gỗ** dài `S` đơn vị, cắt thành các đoạn nối đuôi: đoạn của chỉ số `i` dài `w[i]`. Bạn **nhắm mắt chấm một điểm ngẫu nhiên đều** trên thanh. Điểm rơi vào đoạn nào thì chọn chỉ số đó. Đoạn dài gấp 3 thì khả năng bị chấm gấp 3. Vậy xác suất đúng bằng `độ dài / S = w[i] / S`.

Làm thủ công với `w = [1, 3]`, `S = 4`. Ta xem thanh gỗ là 4 "ô" đánh số `1, 2, 3, 4`:

```
ô:      1 | 2 3 4
chỉ số: 0 | 1 1 1
```

Chọn đều một ô trong `1..4` rồi tra bảng: ô `1` → chỉ số `0` (xác suất `1/4`), ô `2, 3, 4` → chỉ số `1` (xác suất `3/4`). Ví dụ lớn hơn `w = [3, 1, 2, 4]`, `S = 10`:

```
ô:      1 2 3 | 4 | 5 6 | 7 8 9 10
chỉ số: 0 0 0 | 1 | 2 2 | 3 3 3 3
```

Mỗi chỉ số `i` sở hữu **đúng `w[i]` ô**. Vậy bài toán đã đưa về hai việc rất đơn giản: (a) chọn đều một ô `x` trong `[1, S]`; (b) cho `x`, tìm xem ô đó thuộc chỉ số nào.

In [8]:
def owner_table(w):
    """Bảng ô -> chỉ số, dựng bằng cách nối đuôi các đoạn (cách 'thủ công' nhất)."""
    table = []
    for i, wi in enumerate(w):
        table.extend([i] * wi)
    return table            # table[x-1] là chủ của ô x (ô đánh số từ 1)


for w in ([1, 3], [3, 1, 2, 4]):
    t = owner_table(w)
    print("w =", w, " S =", len(t))
    print("  ô    :", list(range(1, len(t) + 1)))
    print("  chỉ số:", t)
    print("  xác suất:", {i: f"{t.count(i)}/{len(t)}" for i in range(len(w))})

w = [1, 3]  S = 4
  ô    : [1, 2, 3, 4]
  chỉ số: [0, 1, 1, 1]
  xác suất: {0: '1/4', 1: '3/4'}
w = [3, 1, 2, 4]  S = 10
  ô    : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
  chỉ số: [0, 0, 0, 1, 2, 2, 3, 3, 3, 3]
  xác suất: {0: '3/10', 1: '1/10', 2: '2/10', 3: '4/10'}


## 5. Cách 1 — Brute Force: nhân bản chỉ số

**Ý tưởng:** đúng như bảng ở trên. Tạo mảng `table` gồm `w[0]` bản sao của `0`, `w[1]` bản sao của `1`, ... Mỗi lần `pickIndex` chọn đều một phần tử của `table`: `table[random.randrange(len(table))]`.

Code (hàm minh hoạ, nhận `rng` để chạy có seed):

In [9]:
class ExpandPicker:
    """Cách 1: nhân bản chỉ số. Dễ hiểu nhất, nhưng tốn bộ nhớ O(S)."""
    def __init__(self, w, rng):
        self.rng = rng
        self.table = []
        for i, wi in enumerate(w):
            self.table.extend([i] * wi)

    def pick(self):
        return self.table[self.rng.randrange(len(self.table))]


picker = ExpandPicker([1, 3], XorShift64(5))
print([picker.pick() for _ in range(12)])        # đa số là 1, thỉnh thoảng 0

picker = ExpandPicker([3, 1, 2, 4], XorShift64(5))
cnt = Counter(picker.pick() for _ in range(10000))
print({i: cnt[i] for i in range(4)}, "(kỳ vọng xấp xỉ 3000, 1000, 2000, 4000)")

[0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1]
{0: 2961, 1: 1034, 2: 1991, 3: 4014} (kỳ vọng xấp xỉ 3000, 1000, 2000, 4000)


**Độ phức tạp:** khởi tạo `O(S)` thời gian và **`O(S)` bộ nhớ**; mỗi lần pick `O(1)`.

**Nút thắt:** `S` có thể tới `10^9`. Một list Python tốn khoảng 8 byte mỗi con trỏ, nghĩa là ~8 GB chỉ cho `table`: không thể. Thậm chí chỉ `w = [10**5] * 10**4` đã là một tỉ phần tử. Ta tính thử để thấy rõ:

In [10]:
w_worst = [10 ** 5] * 10 ** 4
S = sum(w_worst)
print("S =", S)
print("Bộ nhớ ước tính cho table: ~", S * 8 / 1e9, "GB (mỗi con trỏ 8 byte)")
print("Bộ nhớ của mảng tiền tố   : ~", len(w_worst) * 8 / 1e3, "KB")

S = 1000000000
Bộ nhớ ước tính cho table: ~ 8.0 GB (mỗi con trỏ 8 byte)
Bộ nhớ của mảng tiền tố   : ~ 80.0 KB


## 6. Ý tưởng thuật toán / bước đột phá

Nhìn lại bảng ô → chỉ số: các ô của cùng một chỉ số nằm **liền nhau**. Vậy ta không cần lưu từng ô, chỉ cần lưu **ô cuối cùng của mỗi đoạn**. Đó chính là mảng tổng tiền tố!

```
w      = [3, 1, 2, 4]
prefix = [3, 4, 6, 10]
```

Chỉ số `i` sở hữu các ô `prefix[i-1] + 1 ... prefix[i]` (với `prefix[-1] := 0`). Cho ô `x`, chỉ số cần tìm là **chỉ số `i` nhỏ nhất thoả `prefix[i] >= x`**.

Kiểm tra: `x = 5`: `prefix[0]=3 < 5`, `prefix[1]=4 < 5`, `prefix[2]=6 >= 5` → chỉ số `2`. Đúng với bảng. Vì `prefix` đã sắp xếp nên "chỉ số nhỏ nhất thoả điều kiện" là bài **tìm kiếm nhị phân** (`bisect_left`).

Bộ nhớ chỉ còn `O(n)`, mỗi lần pick `O(log n)`. Cell sau kiểm chứng đây là **cùng một ánh xạ** với bảng nhân bản:

In [11]:
def locate_bisect(prefix, x):
    """Chỉ số nhỏ nhất i thoả prefix[i] >= x."""
    return bisect.bisect_left(prefix, x)


for w in ([1, 3], [3, 1, 2, 4], [5], [2, 2, 2]):
    prefix = list(accumulate(w))
    table = owner_table(w)
    mapped = [locate_bisect(prefix, x) for x in range(1, prefix[-1] + 1)]
    print(w, "prefix =", prefix, "| trùng bảng nhân bản:", mapped == table)
    assert mapped == table

[1, 3] prefix = [1, 4] | trùng bảng nhân bản: True
[3, 1, 2, 4] prefix = [3, 4, 6, 10] | trùng bảng nhân bản: True
[5] prefix = [5] | trùng bảng nhân bản: True
[2, 2, 2] prefix = [2, 4, 6] | trùng bảng nhân bản: True


## 7. Các cách tiếp theo

Ta xét thêm 3 cách (cùng Cách 1 ở mục 5 là 4 cách): **Cách 2** tổng tiền tố + quét tuyến tính, **Cách 3** tổng tiền tố + tìm kiếm nhị phân (cách chuẩn), **Cách 4** phương pháp alias.

### Cách 2 — Tổng tiền tố + quét tuyến tính

**Giải thích từng đoạn:**

1. Dựng `prefix` một lần trong constructor, `O(n)`.
2. Mỗi lần pick: chọn `x = randint(1, S)`.
3. Duyệt `i = 0, 1, 2, ...` và dừng ở chỉ số đầu tiên có `prefix[i] >= x`.

Nó bỏ được bộ nhớ `O(S)` nhưng mỗi lần pick vẫn `O(n)`.

In [12]:
def locate_linear(prefix, x):
    """Quét từ trái: chỉ số đầu tiên có prefix[i] >= x."""
    for i, edge in enumerate(prefix):
        if edge >= x:
            return i
    raise ValueError("x vượt quá tổng")


class LinearPicker:
    def __init__(self, w, rng):
        self.rng = rng
        self.prefix = list(accumulate(w))

    def pick(self):
        x = self.rng.randint(1, self.prefix[-1])
        return locate_linear(self.prefix, x)


pk = LinearPicker([3, 1, 2, 4], XorShift64(11))
cnt = Counter(pk.pick() for _ in range(10000))
print({i: cnt[i] for i in range(4)})

{0: 2972, 1: 992, 2: 2011, 3: 4025}


In [13]:
# Code nộp LeetCode cho Cách 2
class SolutionLinear:
    def __init__(self, w: List[int]):
        self.prefix = []
        total = 0
        for x in w:
            total += x
            self.prefix.append(total)

    def pickIndex(self) -> int:
        x = random.randint(1, self.prefix[-1])
        for i, edge in enumerate(self.prefix):
            if edge >= x:
                return i
        return len(self.prefix) - 1      # không bao giờ tới đây, để cho trình kiểm tra yên tâm

**Độ phức tạp:** khởi tạo `O(n)`; mỗi `pickIndex` `O(n)`; bộ nhớ `O(n)`. Với `n = 10^4` và `10^4` lần gọi: ~`10^8` bước lặp Python, hơi sát giới hạn. Ta sẽ đo thời gian ở mục 11.

### Cách 3 — Tổng tiền tố + tìm kiếm nhị phân (cách chuẩn)

**Giải thích từng đoạn:**

1. `prefix` đã sắp xếp tăng chặt.
2. Ta cần chỉ số nhỏ nhất `i` có `prefix[i] >= x`. Giữ cửa sổ `[lo, hi]` luôn chứa đáp án; ban đầu `lo = 0`, `hi = n - 1` (đáp án chắc chắn nằm đây vì `x <= S = prefix[n-1]`).
3. Lấy `mid = (lo + hi) // 2`. Nếu `prefix[mid] >= x` thì `mid` có thể là đáp án hoặc đáp án nằm bên trái → `hi = mid`. Ngược lại đáp án ở bên phải `mid` → `lo = mid + 1`.
4. Lặp tới khi `lo == hi`; đó là đáp án.

Cố ý dùng `hi = mid` (không phải `mid - 1`) vì `mid` vẫn còn khả năng là đáp án.

In [14]:
def locate_binary(prefix, x):
    lo, hi = 0, len(prefix) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if prefix[mid] >= x:
            hi = mid            # mid có thể là đáp án -> giữ lại
        else:
            lo = mid + 1        # mid chắc chắn nhỏ quá -> loại
    return lo


class BinaryPicker:
    def __init__(self, w, rng):
        self.rng = rng
        self.prefix = list(accumulate(w))

    def pick(self):
        x = self.rng.randint(1, self.prefix[-1])
        return locate_binary(self.prefix, x)


pk = BinaryPicker([3, 1, 2, 4], XorShift64(11))
cnt = Counter(pk.pick() for _ in range(10000))
print({i: cnt[i] for i in range(4)})

# locate_binary phải trùng locate_bisect (thư viện) và locate_linear ở MỌI x
for w in ([1], [1, 3], [3, 1, 2, 4], [7, 7, 7, 7, 7], [100000, 1, 100000]):
    prefix = list(accumulate(w))
    for x in range(1, min(prefix[-1], 5000) + 1):
        assert locate_binary(prefix, x) == locate_bisect(prefix, x) == locate_linear(prefix, x)
print("3 hàm định vị cho cùng kết quả")

{0: 2972, 1: 992, 2: 2011, 3: 4025}
3 hàm định vị cho cùng kết quả


In [15]:
# Code nộp LeetCode cho Cách 3 (tự viết nhị phân)
class SolutionBinary:
    def __init__(self, w: List[int]):
        self.prefix = []
        total = 0
        for x in w:
            total += x
            self.prefix.append(total)

    def pickIndex(self) -> int:
        x = random.randint(1, self.prefix[-1])
        lo, hi = 0, len(self.prefix) - 1
        while lo < hi:
            mid = (lo + hi) // 2
            if self.prefix[mid] >= x:
                hi = mid
            else:
                lo = mid + 1
        return lo


# Biến thể dùng thư viện: chỉ 1 dòng thay cho vòng while
class SolutionBisect:
    def __init__(self, w: List[int]):
        self.prefix = list(accumulate(w))

    def pickIndex(self) -> int:
        return bisect.bisect_left(self.prefix, random.randint(1, self.prefix[-1]))

**Độ phức tạp:** khởi tạo `O(n)`; mỗi `pickIndex` `O(log n)`; bộ nhớ `O(n)`.

### Cách 4 — Phương pháp alias (Walker / Vose), chọn trong O(1)

**Ý tưởng:** có thể còn nhanh hơn `O(log n)` không? Có, với giá của một bước tiền xử lý khéo. Chia "thanh gỗ" thành `n` **cột bằng nhau**, mỗi cột chứa **tối đa hai chỉ số**: chủ cột và một "kẻ đi mượn" (alias). Khi chọn:

1. Chọn đều một cột `j` trong `n` cột.
2. Tung "đồng xu lệch" trong cột: với xác suất `thr[j] / S` lấy chủ cột `j`, còn lại lấy `alias[j]`.

Mọi thứ làm bằng số nguyên để chính xác. Mỗi cột có sức chứa `S` ô; chỉ số `i` có tổng `w[i] · n` đơn vị (vì có `n` cột), và tổng cộng `n · S`, đúng bằng sức chứa của `n` cột. Dựng bảng:

1. `scaled[i] = w[i] * n`. Chia chỉ số thành nhóm **nhỏ** (`scaled < S`) và nhóm **lớn** (`scaled >= S`).
2. Lấy một chỉ số nhỏ `s` và một chỉ số lớn `l`. Cột `s` có `thr[s] = scaled[s]` đơn vị của chính `s`, phần còn lại `S - scaled[s]` mượn từ `l` (`alias[s] = l`). Trừ `scaled[l] -= S - scaled[s]`; `l` có thể thành nhỏ → đưa về nhóm nhỏ.
3. Phần còn dư (làm tròn) tự chiếm trọn cột: `thr = S`.

Mỗi lần pick tốn đúng **hai** lần gọi random và vài phép so sánh: `O(1)`.

In [16]:
def build_alias(w, verbose=False):
    """Dựng bảng alias bằng số nguyên. Trả (thr, alias, S)."""
    n, S = len(w), sum(w)
    scaled = [wi * n for wi in w]                 # tổng = n*S, mỗi cột chứa S
    small = [i for i in range(n) if scaled[i] < S]
    large = [i for i in range(n) if scaled[i] >= S]
    thr, alias = [S] * n, list(range(n))
    while small and large:
        s, l = small.pop(), large.pop()
        thr[s], alias[s] = scaled[s], l            # cột s: scaled[s] đơn vị của s, phần còn lại của l
        scaled[l] -= S - scaled[s]
        if verbose:
            print(f"  cột {s}: thr={thr[s]:>3}, alias={l}; chỉ số {l} còn {scaled[l]}")
        (small if scaled[l] < S else large).append(l)
    return thr, alias, S


def alias_pick(thr, alias, S, rng):
    j = rng.randrange(len(thr))                    # chọn cột
    r = rng.randrange(S)                           # đồng xu lệch: r thuộc [0, S)
    return j if r < thr[j] else alias[j]


thr, alias, S = build_alias([3, 1, 2, 4], verbose=True)
print("thr =", thr, " alias =", alias, " S =", S)
rng = XorShift64(3)
cnt = Counter(alias_pick(thr, alias, S, rng) for _ in range(10000))
print({i: cnt[i] for i in range(4)}, "(kỳ vọng ~3000, 1000, 2000, 4000)")

  cột 2: thr=  8, alias=3; chỉ số 3 còn 14
  cột 1: thr=  4, alias=3; chỉ số 3 còn 8
  cột 3: thr=  8, alias=0; chỉ số 0 còn 10
thr = [10, 4, 8, 8]  alias = [0, 3, 3, 0]  S = 10
{0: 2909, 1: 1060, 2: 2029, 3: 4002} (kỳ vọng ~3000, 1000, 2000, 4000)


In [17]:
# Code nộp LeetCode cho Cách 4
class SolutionAlias:
    def __init__(self, w: List[int]):
        n, total = len(w), sum(w)
        scaled = [x * n for x in w]
        small = [i for i in range(n) if scaled[i] < total]
        large = [i for i in range(n) if scaled[i] >= total]
        self.n, self.total = n, total
        self.thr = [total] * n
        self.alias = list(range(n))
        while small and large:
            s, l = small.pop(), large.pop()
            self.thr[s], self.alias[s] = scaled[s], l
            scaled[l] -= total - scaled[s]
            (small if scaled[l] < total else large).append(l)

    def pickIndex(self) -> int:
        j = random.randrange(self.n)
        r = random.randrange(self.total)
        return j if r < self.thr[j] else self.alias[j]

**Độ phức tạp:** dựng bảng `O(n)`; mỗi `pickIndex` `O(1)`; bộ nhớ `O(n)`. Chứng minh xác suất đúng bằng `Fraction` ở mục 10 và mục 12.

**Khi nào dùng alias?** Khi số lần pick **cực lớn** so với `n` và tốc độ quan trọng. Với bài này, nhị phân đã quá đủ nhanh và *đơn giản hơn nhiều*, nên là câu trả lời chuẩn khi phỏng vấn; alias là kiến thức mở rộng đáng biết.

## 8. TRACE từng bước

Lấy `w = [3, 1, 2, 4]`. Ta trace ba thứ: (1) tiền tố, (2) tìm kiếm nhị phân cho vài `x`, (3) một lần `pickIndex` đầy đủ với RNG có seed.

**Bảng trạng thái của quá trình dựng tiền tố:**

| bước | `x = w[i]` | `total` sau khi cộng | `prefix` |
|---|---|---|---|
| 0 | — | 0 | `[]` |
| 1 | 3 | 3 | `[3]` |
| 2 | 1 | 4 | `[3, 4]` |
| 3 | 2 | 6 | `[3, 4, 6]` |
| 4 | 4 | 10 | `[3, 4, 6, 10]` |

**Bảng nhị phân cho `x = 5`** (`prefix = [3, 4, 6, 10]`, đang tìm chỉ số nhỏ nhất có `prefix[i] >= 5`):

| vòng | `lo` | `hi` | `mid` | `prefix[mid]` | so với `x = 5` | hành động |
|---|---|---|---|---|---|---|
| 1 | 0 | 3 | 1 | 4 | `4 < 5` | `lo = mid + 1 = 2` |
| 2 | 2 | 3 | 2 | 6 | `6 >= 5` | `hi = mid = 2` |
| dừng | 2 | 2 | — | — | `lo == hi` | trả về `2` |

In [18]:
def trace_prefix(w):
    total, prefix = 0, []
    print(f"{'bước':>4} {'x':>3} {'total':>5}  prefix")
    print(f"{0:>4} {'-':>3} {0:>5}  {prefix}")
    for step, x in enumerate(w, 1):
        total += x
        prefix.append(total)
        print(f"{step:>4} {x:>3} {total:>5}  {prefix}")
    return prefix


def trace_binary(prefix, x):
    print(f"\nTìm chỉ số nhỏ nhất có prefix[i] >= {x}   (prefix = {prefix})")
    lo, hi = 0, len(prefix) - 1
    vong = 0
    while lo < hi:
        vong += 1
        mid = (lo + hi) // 2
        if prefix[mid] >= x:
            action = f"{prefix[mid]} >= {x} -> hi = mid = {mid}"
            hi = mid
        else:
            action = f"{prefix[mid]} <  {x} -> lo = mid + 1 = {mid + 1}"
            lo = mid + 1
        print(f"  vòng {vong}: mid={mid} | {action} | cửa sổ mới [{lo}, {hi}]")
    print(f"  dừng: lo == hi == {lo}")
    return lo


prefix = trace_prefix([3, 1, 2, 4])
for x in (1, 4, 5, 10):
    ans = trace_binary(prefix, x)
    assert ans == bisect.bisect_left(prefix, x)

bước   x total  prefix
   0   -     0  []
   1   3     3  [3]
   2   1     4  [3, 4]
   3   2     6  [3, 4, 6]
   4   4    10  [3, 4, 6, 10]

Tìm chỉ số nhỏ nhất có prefix[i] >= 1   (prefix = [3, 4, 6, 10])
  vòng 1: mid=1 | 4 >= 1 -> hi = mid = 1 | cửa sổ mới [0, 1]
  vòng 2: mid=0 | 3 >= 1 -> hi = mid = 0 | cửa sổ mới [0, 0]
  dừng: lo == hi == 0

Tìm chỉ số nhỏ nhất có prefix[i] >= 4   (prefix = [3, 4, 6, 10])
  vòng 1: mid=1 | 4 >= 4 -> hi = mid = 1 | cửa sổ mới [0, 1]
  vòng 2: mid=0 | 3 <  4 -> lo = mid + 1 = 1 | cửa sổ mới [1, 1]
  dừng: lo == hi == 1

Tìm chỉ số nhỏ nhất có prefix[i] >= 5   (prefix = [3, 4, 6, 10])
  vòng 1: mid=1 | 4 <  5 -> lo = mid + 1 = 2 | cửa sổ mới [2, 3]
  vòng 2: mid=2 | 6 >= 5 -> hi = mid = 2 | cửa sổ mới [2, 2]
  dừng: lo == hi == 2

Tìm chỉ số nhỏ nhất có prefix[i] >= 10   (prefix = [3, 4, 6, 10])
  vòng 1: mid=1 | 4 <  10 -> lo = mid + 1 = 2 | cửa sổ mới [2, 3]
  vòng 2: mid=2 | 6 <  10 -> lo = mid + 1 = 3 | cửa sổ mới [3, 3]
  dừng: lo == hi == 3


In [19]:
def trace_pick(w, rng, times=4):
    """Một lần pickIndex đầy đủ: bốc x bằng rng rồi định vị, in từng bước."""
    prefix = list(accumulate(w))
    S = prefix[-1]
    print("w =", w, " prefix =", prefix, " S =", S)
    for t in range(1, times + 1):
        x = rng.randint(1, S)
        i = bisect.bisect_left(prefix, x)
        lower = prefix[i - 1] + 1 if i else 1
        print(f"  lần {t}: x = {x:>2} -> chỉ số {i} (đoạn sở hữu các ô {lower}..{prefix[i]})")


trace_pick([3, 1, 2, 4], XorShift64(99))

w = [3, 1, 2, 4]  prefix = [3, 4, 6, 10]  S = 10
  lần 1: x =  1 -> chỉ số 0 (đoạn sở hữu các ô 1..3)
  lần 2: x =  6 -> chỉ số 2 (đoạn sở hữu các ô 5..6)
  lần 3: x =  8 -> chỉ số 3 (đoạn sở hữu các ô 7..10)
  lần 4: x =  5 -> chỉ số 2 (đoạn sở hữu các ô 5..6)


**Trace phần dựng bảng alias** cho `w = [3, 1, 2, 4]` (`n = 4`, `S = 10`, mỗi cột chứa 10 ô):

| bước | thao tác | `thr` / `alias` của cột vừa chốt | `scaled` sau đó | nhóm nhỏ | nhóm lớn |
|---|---|---|---|---|---|
| đầu | — | — | `[12, 4, 8, 16]` | `[1, 2]` | `[0, 3]` |
| 1 | `s = 2`, `l = 3` | cột 2: `thr = 8`, `alias = 3` | `[12, 4, 8, 14]` | `[1]` | `[0, 3]` |
| 2 | `s = 1`, `l = 3` | cột 1: `thr = 4`, `alias = 3` | `[12, 4, 8, 8]` | `[]` → thêm `3` | `[0]` |
| 3 | `s = 3`, `l = 0` | cột 3: `thr = 8`, `alias = 0` | `[10, 4, 8, 8]` | `[]` | `[0]` (chỉ số 0 còn đúng 10) |
| cuối | hết nhóm nhỏ | cột 0: `thr = S = 10` (tự chiếm trọn) | — | — | — |

Thay vì tự đoán bảng, ta cho máy in từng bước (đã làm ở mục 7 với `verbose=True`). Ở đây ta kiểm chứng *kết quả*: xác suất của mỗi chỉ số từ bảng alias, tính **chính xác bằng Fraction**:

In [20]:
def alias_probabilities(w):
    """P(chọn i) = (1/n) * [ thr[i]/S + sum_{j: alias[j]=i, j!=i} (S - thr[j])/S ]."""
    thr, alias, S = build_alias(w)
    n = len(w)
    prob = [Fraction(0)] * n
    for j in range(n):
        prob[j] += Fraction(1, n) * Fraction(thr[j], S)
        if thr[j] < S:
            prob[alias[j]] += Fraction(1, n) * Fraction(S - thr[j], S)
    return prob


w = [3, 1, 2, 4]
thr, alias, S = build_alias(w)
print("cột  thr  alias")
for j in range(len(w)):
    print(f"{j:>3} {thr[j]:>4} {alias[j]:>5}")
prob = alias_probabilities(w)
print("P(i) từ bảng alias:", [str(p) for p in prob])
print("w[i]/S           :", [str(Fraction(x, sum(w))) for x in w])
assert prob == [Fraction(x, sum(w)) for x in w]

cột  thr  alias
  0   10     0
  1    4     3
  2    8     3
  3    8     0
P(i) từ bảng alias: ['3/10', '1/10', '1/5', '2/5']
w[i]/S           : ['3/10', '1/10', '1/5', '2/5']


## 9. Code nộp LeetCode

Cách chuẩn: **tổng tiền tố + tìm kiếm nhị phân**. Giữ nguyên tên `class Solution`, `__init__(self, w)`, `pickIndex(self)`:

In [21]:
import random
from typing import List


class Solution:

    def __init__(self, w: List[int]):
        self.prefix = []              # prefix[i] = w[0] + ... + w[i]
        total = 0
        for x in w:
            total += x
            self.prefix.append(total)

    def pickIndex(self) -> int:
        x = random.randint(1, self.prefix[-1])      # chọn đều một ô trong [1, S]
        lo, hi = 0, len(self.prefix) - 1
        while lo < hi:                               # tìm chỉ số nhỏ nhất có prefix[i] >= x
            mid = (lo + hi) // 2
            if self.prefix[mid] >= x:
                hi = mid
            else:
                lo = mid + 1
        return lo


# Chạy thử đúng kiểu LeetCode
random.seed(2024)
print(run_ops(Solution, ["Solution", "pickIndex"], [[[1]], []]))
out = run_ops(Solution, ["Solution"] + ["pickIndex"] * 5, [[[1, 3]]] + [[]] * 5)
print(out)
assert out[0] is None and all(v in (0, 1) for v in out[1:])

[None, 0]
[None, 1, 1, 1, 1, 1]


## 10. Vì sao đúng

**Định lý.** Với `x` chọn đều trong `[1, S]` thì `P(pickIndex() = i) = w[i] / S`.

**Chứng minh (3 ý):**

1. *Hàm định vị đúng.* Vòng `while` giữ bất biến: **đáp án (chỉ số nhỏ nhất có `prefix[i] >= x`) luôn nằm trong `[lo, hi]`**. Ban đầu đúng vì `prefix[n-1] = S >= x`. Nếu `prefix[mid] >= x`, đáp án `<= mid`; ta đặt `hi = mid`, vẫn chứa đáp án. Nếu `prefix[mid] < x`, mọi chỉ số `<= mid` đều quá nhỏ (do `prefix` tăng); đáp án `> mid`; ta đặt `lo = mid + 1`. Mỗi vòng cửa sổ thu hẹp thật sự (`lo < hi` kéo theo `mid < hi`), nên dừng sau `O(log n)` vòng với `lo == hi` chính là đáp án.
2. *Mỗi chỉ số sở hữu đúng `w[i]` giá trị `x`.* Chỉ số `i` được trả về khi và chỉ khi `prefix[i-1] < x <= prefix[i]` (với `prefix[-1] := 0`). Số nguyên `x` trong khoảng đó là `prefix[i] − prefix[i−1] = w[i]`.
3. *Vì `x` đều trên `S` giá trị* nên mỗi giá trị xác suất `1/S`; chỉ số `i` có `w[i]` giá trị → xác suất `w[i]/S`. ∎

Đừng tin lời chứng minh chỉ vì nó xuôi tai: ta có thể **kiểm tra bằng máy một cách tuyệt đối**. Ý tưởng: liệt kê *mọi* chuỗi lựa chọn ngẫu nhiên có thể xảy ra (ở đây chỉ là mọi giá trị `x`), chạy `pickIndex` cho từng chuỗi, cộng xác suất — thu được phân phối **chính xác** dạng `Fraction`. Công cụ này tên `exact_distribution`: nó thay `random.randint`/`random.randrange` bằng hàm "đọc kịch bản"; khi kịch bản hết mà code còn xin số ngẫu nhiên, ta rẽ nhánh theo mọi giá trị có thể.

In [22]:
class NeedChoice(Exception):
    """Báo: code đang xin một số ngẫu nhiên trong [lo, hi] nhưng kịch bản đã hết."""
    def __init__(self, lo, hi):
        self.lo, self.hi = lo, hi


@contextlib.contextmanager
def patched_random(draw):
    """Tạm thay random.randint / random.randrange bằng hàm draw(lo, hi) (hai đầu đều lấy được)."""
    old = random.randint, random.randrange
    random.randint = lambda a, b: draw(a, b)
    random.randrange = lambda a, b=None: draw(0, a - 1) if b is None else draw(a, b - 1)
    try:
        yield
    finally:
        random.randint, random.randrange = old


def exact_distribution(fn):
    """Liệt kê MỌI chuỗi lựa chọn của fn() và trả dict {kết quả: xác suất chính xác (Fraction)}.
    Nếu fn ném ngoại lệ, kết quả được ghi là tên ngoại lệ."""
    dist = {}

    def explore(choices, prob):
        pos = [0]

        def draw(lo, hi):
            if pos[0] < len(choices):
                v = choices[pos[0]]
                pos[0] += 1
                return v
            raise NeedChoice(lo, hi)

        try:
            with patched_random(draw):
                res = fn()
        except NeedChoice as need:
            n = need.hi - need.lo + 1
            for v in range(need.lo, need.hi + 1):
                explore(choices + [v], prob / n)
            return
        except Exception as e:
            res = type(e).__name__
        dist[res] = dist.get(res, 0) + prob

    explore([], Fraction(1))
    return dist


def exact_picks(cls, w):
    """Phân phối chính xác của cls(w).pickIndex()."""
    obj = cls(w)                       # constructor không được xin số ngẫu nhiên
    return exact_distribution(obj.pickIndex)


def expected(w):
    S = sum(w)
    return {i: Fraction(wi, S) for i, wi in enumerate(w)}


print(exact_picks(Solution, [1, 3]))
print(exact_picks(Solution, [3, 1, 2, 4]))
for cls in (Solution, SolutionLinear, SolutionBinary, SolutionBisect, SolutionAlias):
    for w in ([1], [1, 3], [3, 1, 2, 4], [5, 5, 5], [1, 1, 1, 1, 1, 9], [2, 7]):
        assert exact_picks(cls, w) == expected(w), (cls.__name__, w)
print("5 lớp x 6 mảng: phân phối ĐÚNG TUYỆT ĐỐI (so bằng Fraction)")

{0: Fraction(1, 4), 1: Fraction(3, 4)}
{0: Fraction(3, 10), 1: Fraction(1, 10), 2: Fraction(1, 5), 3: Fraction(2, 5)}
5 lớp x 6 mảng: phân phối ĐÚNG TUYỆT ĐỐI (so bằng Fraction)


## 11. Time / Space Complexity

Đặt `n = len(w)`, `S = sum(w)`.

| Cách | Khởi tạo | Mỗi `pickIndex` | Bộ nhớ |
|---|---|---|---|
| 1. Nhân bản chỉ số | `O(S)` | `O(1)` | **`O(S)`** (tới `10^9`) |
| 2. Prefix + quét tuyến tính | `O(n)` | `O(n)` | `O(n)` |
| 3. Prefix + nhị phân | `O(n)` | `O(log n)` | `O(n)` |
| 4. Alias | `O(n)` | `O(1)` | `O(n)` |

Với giới hạn đề (`n, số lần gọi <= 10^4`): Cách 2 tốn tối đa ~`10^8` thao tác; Cách 3 chỉ ~`1.4·10^5`. Hãy đo thực tế:

In [23]:
def bench(cls, w, calls, seed=1):
    random.seed(seed)
    obj = cls(w)
    t0 = time.perf_counter()
    for _ in range(calls):
        obj.pickIndex()
    return time.perf_counter() - t0


gen = XorShift64(77)
n = 2000
w_big = [gen.randint(1, 10 ** 5) for _ in range(n)]
print(f"n = {n}, 2000 lần pickIndex, tổng S = {sum(w_big)}")
for cls in (SolutionLinear, SolutionBinary, SolutionBisect, SolutionAlias):
    print(f"  {cls.__name__:15s} {bench(cls, w_big, 2000) * 1000:8.2f} ms")
print("(thời gian chỉ mang tính tham khảo; quan trọng là Linear chậm rõ rệt khi n lớn)")

n = 2000, 2000 lần pickIndex, tổng S = 98599051


  SolutionLinear     62.35 ms
  SolutionBinary      4.02 ms
  SolutionBisect      1.81 ms
  SolutionAlias       1.09 ms
(thời gian chỉ mang tính tham khảo; quan trọng là Linear chậm rõ rệt khi n lớn)


## 12. Bộ test

Bộ test chia thành 6 nhóm: (A) ví dụ của đề, (B) edge case, (C) **đếm chính xác bằng `Fraction`**, (D) phản ví dụ — các cài đặt sai *phải bị bắt*, (E) đối chiếu các cách bằng `assert` trên nhiều input, (F) **thống kê chi-bình-phương** có seed trên input đúng ràng buộc.

### A. Ví dụ của đề

In [24]:
# Ví dụ 1: w = [1] -> luôn 0
for seed in range(20):
    random.seed(seed)
    assert run_ops(Solution, ["Solution", "pickIndex"], [[[1]], []]) == [None, 0]

# Ví dụ 2: w = [1, 3] -> mọi đáp án chỉ gồm 0 hoặc 1; tỷ lệ 1 xấp xỉ 75%
random.seed(2024)
res = run_ops(Solution, ["Solution"] + ["pickIndex"] * 4000, [[[1, 3]]] + [[]] * 4000)
vals = res[1:]
print("Tỷ lệ chọn chỉ số 1:", vals.count(1) / len(vals), "(kỳ vọng 0.75)")
assert set(vals) <= {0, 1}
assert abs(vals.count(1) / len(vals) - 0.75) < 0.03

Tỷ lệ chọn chỉ số 1: 0.7415 (kỳ vọng 0.75)


### B. Edge case

- `n = 1`: luôn trả `0`.
- Mọi trọng số bằng nhau: phân phối đều.
- Một trọng số khổng lồ cạnh trọng số nhỏ (`[1, 10**5]`).
- Ràng buộc lớn nhất: `n = 10^4`, mỗi `w[i] = 10^5` (`S = 10^9`) — cách nhân bản sẽ chết, cách tiền tố vẫn mượt.
- Hai đầu: `x = 1` phải ra chỉ số `0`, `x = S` phải ra chỉ số `n - 1`.

In [25]:
# n = 1
assert exact_picks(Solution, [100000]) == {0: 1}

# trọng số bằng nhau
assert exact_picks(Solution, [4, 4, 4, 4]) == {i: Fraction(1, 4) for i in range(4)}

# hai biên x = 1 và x = S
for w in ([3, 1, 2, 4], [1], [1, 1], [10, 1]):
    obj = Solution(w)
    S = sum(w)
    with patched_random(lambda a, b: 1):
        assert obj.pickIndex() == 0
    with patched_random(lambda a, b: S):
        assert obj.pickIndex() == len(w) - 1

# ràng buộc lớn nhất: không được nhân bản, vẫn chạy nhanh
w_max = [10 ** 5] * 10 ** 4
t0 = time.perf_counter()
obj = Solution(w_max)
random.seed(5)
picks = [obj.pickIndex() for _ in range(10 ** 4)]
dt = time.perf_counter() - t0
assert all(0 <= p < 10 ** 4 for p in picks)
print(f"n = 10^4, S = 10^9: dựng + 10^4 lần pick mất {dt:.3f}s; số chỉ số khác nhau thấy được: {len(set(picks))}")

# một đầu mút cực đoan: x = S - 1 và x = S đều thuộc chỉ số cuối khi w[-1] >= 2
obj = Solution([1, 10 ** 5])
with patched_random(lambda a, b: 2):
    assert obj.pickIndex() == 1
print("edge case OK")

n = 10^4, S = 10^9: dựng + 10^4 lần pick mất 0.018s; số chỉ số khác nhau thấy được: 6276
edge case OK


### C. Đếm chính xác bằng `Fraction` (mọi mảng nhỏ)

Ta kiểm tra **mọi** mảng `w` có độ dài `1..4` với phần tử `1..4` (tổng cộng `4 + 16 + 64 + 256 = 340` mảng) cho **cả 5 lớp**. Mỗi mảng chứa khẳng định "xác suất đúng bằng `w[i]/S`" được chứng minh bằng máy, không dựa vào may rủi.

In [26]:
from itertools import product

classes = [Solution, SolutionLinear, SolutionBinary, SolutionBisect, SolutionAlias]
tested = 0
for n in range(1, 5):
    for w in product(range(1, 5), repeat=n):
        w = list(w)
        want = expected(w)
        for cls in classes:
            assert exact_picks(cls, w) == want, (cls.__name__, w)
        tested += 1
print(f"{tested} mảng x {len(classes)} lớp: tất cả đúng tuyệt đối")

340 mảng x 5 lớp: tất cả đúng tuyệt đối


### D. Phản ví dụ: các cài đặt sai phải bị bắt

Một bộ test tốt phải **thất bại** khi code sai. Ta viết 4 cài đặt có lỗi và cho `exact_picks` "vạch mặt" chúng (mục 13 sẽ giải thích chi tiết từng lỗi).

In [27]:
class BugOffByOne:                   # x thuộc [0, S] (S+1 giá trị) thay vì [1, S]
    def __init__(self, w):
        self.prefix = list(accumulate(w))
    def pickIndex(self):
        return bisect.bisect_left(self.prefix, random.randint(0, self.prefix[-1]))


class BugBisectRight:                # bisect_right với x thuộc [1, S]
    def __init__(self, w):
        self.prefix = list(accumulate(w))
    def pickIndex(self):
        return bisect.bisect_right(self.prefix, random.randint(1, self.prefix[-1]))


class BugUniform:                    # bỏ qua trọng số
    def __init__(self, w):
        self.n = len(w)
    def pickIndex(self):
        return random.randint(0, self.n - 1)


class BugZeroBased:                  # x thuộc [0, S-1] nhưng vẫn dùng bisect_left
    def __init__(self, w):
        self.prefix = list(accumulate(w))
    def pickIndex(self):
        return bisect.bisect_left(self.prefix, random.randint(0, self.prefix[-1] - 1))


w = [1, 3]
for bug in (BugOffByOne, BugBisectRight, BugUniform, BugZeroBased):
    got = exact_picks(bug, w)
    print(f"{bug.__name__:15s} -> {({k: str(v) for k, v in got.items()})}  (đúng phải là 0:1/4, 1:3/4)")
    assert got != expected(w), bug.__name__
print("Cả 4 cài đặt sai đều bị bắt")

BugOffByOne     -> {0: '2/5', 1: '3/5'}  (đúng phải là 0:1/4, 1:3/4)
BugBisectRight  -> {1: '3/4', 2: '1/4'}  (đúng phải là 0:1/4, 1:3/4)
BugUniform      -> {0: '1/2', 1: '1/2'}  (đúng phải là 0:1/4, 1:3/4)
BugZeroBased    -> {0: '1/2', 1: '1/2'}  (đúng phải là 0:1/4, 1:3/4)
Cả 4 cài đặt sai đều bị bắt


### E. Đối chiếu các cách bằng `assert`

Hai kiểm chứng độc lập:

1. **Cùng một `x` thì mọi cách định vị cho cùng một chỉ số** (so sánh `locate_linear`, `locate_binary`, `bisect_left` và bảng nhân bản làm *oracle* vét cạn) trên nhiều mảng ngẫu nhiên có seed.
2. **Alias**: phân phối tính từ bảng bằng `Fraction` phải bằng `w[i]/S` trên 300 mảng ngẫu nhiên có độ dài tới 12.

In [28]:
rng = XorShift64(31415)
checked = 0
for _ in range(300):
    n = rng.randint(1, 12)
    w = [rng.randint(1, 9) for _ in range(n)]
    prefix = list(accumulate(w))
    oracle = owner_table(w)                      # vét cạn: bảng nhân bản
    for x in range(1, prefix[-1] + 1):
        a = oracle[x - 1]
        assert locate_linear(prefix, x) == a
        assert locate_binary(prefix, x) == a
        assert locate_bisect(prefix, x) == a
        checked += 1
    assert alias_probabilities(w) == [Fraction(v, sum(w)) for v in w]
print(f"{checked} cặp (mảng, x) khớp giữa 4 cách; alias đúng trên 300 mảng")

# lớp nộp bài đúng với cả mảng dài hơn (đếm chính xác, n tới 12)
rng = XorShift64(2718)
for _ in range(25):
    n = rng.randint(5, 12)
    w = [rng.randint(1, 6) for _ in range(n)]
    for cls in (Solution, SolutionAlias):
        assert exact_picks(cls, w) == expected(w)
print("đếm chính xác trên 25 mảng dài cho Solution và SolutionAlias: OK")

10071 cặp (mảng, x) khớp giữa 4 cách; alias đúng trên 300 mảng


đếm chính xác trên 25 mảng dài cho Solution và SolutionAlias: OK


### F. Thống kê chi-bình-phương có seed

Test thống kê chỉ cho **bằng chứng xác suất**, nhưng bắt được những lỗi mà ta không nghĩ tới và chạy được với mảng lớn đúng ràng buộc. Quy ước: `N` lần pick, yêu cầu mọi kỳ vọng `E_i >= 5`, và code đúng khi p-value `> 0.001`. Vì dùng seed cố định nên kết quả **lặp lại y hệt** mỗi lần chạy.

In [29]:
def sample_counts(cls, w, N, seed):
    random.seed(seed)
    obj = cls(w)
    cnt = [0] * len(w)
    for _ in range(N):
        cnt[obj.pickIndex()] += 1
    return cnt


def stat_check(cls, w, N, seed, alpha=0.001):
    S = sum(w)
    assert min(N * x / S for x in w) >= 5, "kỳ vọng quá nhỏ, tăng N"
    cnt = sample_counts(cls, w, N, seed)
    stat, p = chi_square_test(cnt, w)
    return cnt, stat, p, p > alpha


cases = [([1, 3], 40000, 1), ([3, 1, 2, 4], 100000, 2), ([1, 1, 1, 1, 1, 1], 60000, 3),
         ([5, 1, 1, 1, 12, 7, 3], 120000, 4), ([100000, 20000, 50000], 60000, 5)]
for cls in (Solution, SolutionAlias):
    for w, N, seed in cases:
        cnt, stat, p, ok = stat_check(cls, w, N, seed)
        print(f"{cls.__name__:13s} w={w!s:24} chi2={stat:7.3f} p={p:.3f} {'OK' if ok else 'FAIL'}")
        assert ok

Solution      w=[1, 3]                   chi2=  0.043 p=0.835 OK
Solution      w=[3, 1, 2, 4]             chi2=  4.336 p=0.227 OK
Solution      w=[1, 1, 1, 1, 1, 1]       chi2=  5.681 p=0.338 OK


Solution      w=[5, 1, 1, 1, 12, 7, 3]   chi2=  4.975 p=0.547 OK


Solution      w=[100000, 20000, 50000]   chi2=  0.951 p=0.622 OK
SolutionAlias w=[1, 3]                   chi2=  1.613 p=0.204 OK
SolutionAlias w=[3, 1, 2, 4]             chi2=  1.288 p=0.732 OK
SolutionAlias w=[1, 1, 1, 1, 1, 1]       chi2=  3.484 p=0.626 OK
SolutionAlias w=[5, 1, 1, 1, 12, 7, 3]   chi2=  4.731 p=0.579 OK


SolutionAlias w=[100000, 20000, 50000]   chi2=  0.795 p=0.672 OK


In [30]:
# Mảng ngẫu nhiên đúng ràng buộc (n nhỏ để đủ mẫu), kiểm định cho từng mảng
rng = XorShift64(8)
fails = 0
for trial in range(12):
    n = rng.randint(2, 20)
    w = [rng.randint(1, 100000) for _ in range(n)]
    N = int(5 * sum(w) / min(w)) + 1000        # đủ để E_min >= 5
    N = min(N, 400000)
    if min(N * x / sum(w) for x in w) < 5:
        continue
    cnt, stat, p, ok = stat_check(Solution, w, N, seed=trial)
    fails += not ok
    print(f"thử {trial:2d}: n={n:2d} N={N:6d} chi2={stat:8.2f} p={p:.3f} {'OK' if ok else 'FAIL'}")
assert fails == 0

# Sức mạnh của test: cài đặt sai BugUniform và BugOffByOne phải bị loại (p cực nhỏ)
for bug, w, N in ((BugUniform, [1, 3], 20000), (BugOffByOne, [1, 1, 1, 1], 100000)):
    cnt, stat, p, ok = stat_check(bug, w, N, seed=1)
    print(f"{bug.__name__:13s} w={w} đếm={cnt} chi2={stat:.1f} p={p:.2e} -> {'BỊ BẮT' if not ok else 'lọt lưới'}")
    assert not ok

thử  0: n=19 N=  2185 chi2=   15.25 p=0.645 OK
thử  1: n= 9 N=  1165 chi2=    8.03 p=0.431 OK
thử  2: n=14 N=  1291 chi2=   13.34 p=0.422 OK
thử  3: n=17 N=  3594 chi2=   14.61 p=0.553 OK
thử  4: n= 3 N=  1170 chi2=    3.77 p=0.152 OK
thử  5: n= 9 N=  1173 chi2=    3.46 p=0.902 OK
thử  6: n= 4 N=  1026 chi2=    1.60 p=0.659 OK
thử  7: n= 2 N=  1080 chi2=    5.14 p=0.023 OK
thử  8: n= 9 N=  1128 chi2=    9.35 p=0.313 OK
thử  9: n=10 N=  1177 chi2=    8.70 p=0.466 OK
thử 10: n= 9 N=  1196 chi2=   15.73 p=0.046 OK
thử 11: n= 6 N=  1054 chi2=    2.46 p=0.783 OK
BugUniform    w=[1, 3] đếm=[10045, 9955] chi2=6787.2 p=0.00e+00 -> BỊ BẮT


BugOffByOne   w=[1, 1, 1, 1] đếm=[39929, 19784, 20238, 20049] chi2=11890.8 p=0.00e+00 -> BỊ BẮT


Vài lưu ý khi test bằng thống kê:

- Test thống kê **không bao giờ chứng minh được 100%**; với ngưỡng `0.001`, code đúng vẫn có `1/1000` bị "báo động giả" nếu đổi seed liên tục. Seed cố định giúp kết quả ổn định, và **đếm chính xác (nhóm C)** mới là bằng chứng cứng.
- Phải kiểm tra điều kiện `E_i >= 5`: nếu kỳ vọng nhỏ, xấp xỉ chi-bình-phương không còn đáng tin.
- Kiểm tra cả "test phải bắt được code sai" (nhóm D và cell trên) để biết bài test có đủ độ nhạy.

## 13. Các lỗi người mới hay gặp

### Lỗi 1: `randint(0, S)` thay vì `randint(1, S)` (lệch 1 ở biên)

`random.randint(a, b)` lấy được **cả hai đầu**. Chọn `x` trong `[0, S]` thì có `S + 1` giá trị; giá trị `0` rơi vào chỉ số `0` (vì `prefix[0] >= 0`), làm chỉ số `0` có thêm một ô.

In [31]:
# CODE SAI
class Sai1:
    def __init__(self, w):
        self.prefix = list(accumulate(w))
    def pickIndex(self):
        x = random.randint(0, self.prefix[-1])           # SAI: S+1 giá trị
        return bisect.bisect_left(self.prefix, x)


print("w = [1, 3]: đúng phải là {0: 1/4, 1: 3/4}")
print("Sai1 cho  :", {k: str(v) for k, v in exact_picks(Sai1, [1, 3]).items()})
print("Chỉ số 0 được 2/5 = 40% thay vì 25%!")

# CODE ĐÚNG
class Dung1:
    def __init__(self, w):
        self.prefix = list(accumulate(w))
    def pickIndex(self):
        x = random.randint(1, self.prefix[-1])           # ĐÚNG: đúng S giá trị
        return bisect.bisect_left(self.prefix, x)


assert exact_picks(Dung1, [1, 3]) == expected([1, 3])

w = [1, 3]: đúng phải là {0: 1/4, 1: 3/4}
Sai1 cho  : {0: '2/5', 1: '3/5'}
Chỉ số 0 được 2/5 = 40% thay vì 25%!


**Vì sao sai:** số ô không còn bằng tổng trọng số. Quy tắc nhớ: *"`randint(1, S)` + `bisect_left`"* hoặc *"`randrange(S)` (tức `[0, S-1]`) + `bisect_right`"* — hai cặp này khớp nhau.

### Lỗi 2: `bisect_right` với `x` trong `[1, S]` (văng lỗi hoặc lệch)

`bisect_right(prefix, x)` trả chỉ số nhỏ nhất có `prefix[i] > x` (**chặt**). Khi `x = S`, không phần tử nào `> S` nên trả `len(prefix)` → chỉ số ra ngoài mảng. Ngoài ra mọi `x` trùng đúng một mút bị nhảy sang chỉ số kế tiếp.

In [32]:
# CODE SAI
class Sai2:
    def __init__(self, w):
        self.prefix = list(accumulate(w))
    def pickIndex(self):
        return bisect.bisect_right(self.prefix, random.randint(1, self.prefix[-1]))   # SAI


print("Sai2 cho w=[1,3]:", {k: str(v) for k, v in exact_picks(Sai2, [1, 3]).items()})
print("-> chỉ số 0 không bao giờ được chọn, còn chỉ số 2 (không tồn tại, w chỉ có chỉ số 0 và 1) xuất hiện với xác suất 1/4")

# CODE ĐÚNG: hoặc đổi sang bisect_left, hoặc dịch x xuống [0, S-1]
class Dung2:
    def __init__(self, w):
        self.prefix = list(accumulate(w))
    def pickIndex(self):
        return bisect.bisect_right(self.prefix, random.randrange(self.prefix[-1]))    # x thuộc [0, S-1]


assert exact_picks(Dung2, [1, 3]) == expected([1, 3])
assert exact_picks(Dung2, [3, 1, 2, 4]) == expected([3, 1, 2, 4])

Sai2 cho w=[1,3]: {1: '3/4', 2: '1/4'}
-> chỉ số 0 không bao giờ được chọn, còn chỉ số 2 (không tồn tại, w chỉ có chỉ số 0 và 1) xuất hiện với xác suất 1/4


### Lỗi 3: Quên trọng số — chọn đều theo chỉ số

Nhiều người đọc nhanh thành "chọn ngẫu nhiên một chỉ số" và viết `random.randint(0, n - 1)`. Code chạy không lỗi nhưng mọi chỉ số bằng xác suất nhau, bỏ qua `w`.

In [33]:
# CODE SAI
class Sai3:
    def __init__(self, w):
        self.n = len(w)
    def pickIndex(self):
        return random.randint(0, self.n - 1)           # SAI: bỏ qua trọng số


print("Sai3 cho w=[1,3]:", {k: str(v) for k, v in exact_picks(Sai3, [1, 3]).items()})
cnt, stat, p, ok = stat_check(Sai3, [1, 3], 20000, 1)
print(f"chi-bình-phương: p = {p:.2e} -> phát hiện được")

# CODE ĐÚNG: tham khảo thư viện chuẩn (cùng ý tưởng cum_weights + bisect bên trong)
class Dung3:
    def __init__(self, w):
        self.idx = list(range(len(w)))
        self.cum = list(accumulate(w))
    def pickIndex(self):
        return random.choices(self.idx, cum_weights=self.cum)[0]


cnt, stat, p, ok = stat_check(Dung3, [1, 3], 20000, 1)
print(f"Dung3: đếm={cnt} p={p:.3f}", "OK" if ok else "FAIL")
assert ok

Sai3 cho w=[1,3]: {0: '1/2', 1: '1/2'}
chi-bình-phương: p = 0.00e+00 -> phát hiện được
Dung3: đếm=[5046, 14954] p=0.453 OK


### Lỗi 4: Dựng lại tiền tố ở mỗi lần `pickIndex` (chậm)

Nếu tính `accumulate(w)` trong `pickIndex`, mỗi lần gọi tốn `O(n)`; với `n = 10^4` và `10^4` lần gọi sẽ rất chậm (và không cần thiết). Nguyên tắc thiết kế lớp: **việc nặng làm một lần ở `__init__`, việc nhẹ làm ở method được gọi nhiều lần**.

In [34]:
# CODE SAI về hiệu năng (vẫn đúng xác suất!)
class Sai4:
    def __init__(self, w):
        self.w = w
    def pickIndex(self):
        prefix = list(accumulate(self.w))                 # SAI: dựng lại mỗi lần gọi
        return bisect.bisect_left(prefix, random.randint(1, prefix[-1]))


w_test = [XorShift64(1).randint(1, 100000) for _ in range(3000)]
print(f"Sai4  (dựng lại)    : {bench(Sai4, w_test, 1000) * 1000:8.1f} ms cho 1000 lần gọi")
print(f"Solution (dựng 1 lần): {bench(Solution, w_test, 1000) * 1000:8.1f} ms cho 1000 lần gọi")
assert exact_picks(Sai4, [1, 3]) == expected([1, 3])        # đúng xác suất, chỉ chậm

Sai4  (dựng lại)    :     73.3 ms cho 1000 lần gọi
Solution (dựng 1 lần):      1.5 ms cho 1000 lần gọi


### Lỗi 5 (bonus): dùng số thực và so sánh bằng `==`

Có người viết `x = random.random() * S` rồi so sánh `x <= prefix[i]`. Cách này **dùng được** (xác suất một giá trị thực trùng đúng mút gần như bằng 0) nhưng bạn không thể chứng minh bằng đếm chính xác, và sai số dấu phẩy động có thể làm `x` bằng hoặc vượt `S` nếu code khác đi một chút. Với bài số nguyên, hãy dùng số nguyên để mọi thứ đếm được.

## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** đề nói "chọn ngẫu nhiên ... với xác suất tỉ lệ với ... (trọng số, diện tích, số lượng)" → **tổng tiền tố + tìm kiếm nhị phân**.

**Công thức chung:**

1. Biến mỗi phần tử thành một **đoạn** có độ dài bằng trọng số.
2. Cộng dồn để có đầu mút (`prefix`).
3. Chọn đều một số nguyên trong `[1, S]`.
4. Tìm đoạn chứa nó bằng `bisect_left`.

**Mẹo kiểm tra nhanh trong đầu:** với `w = [1, 1]` thì `S = 2`, `prefix = [1, 2]`; `x = 1 -> 0`, `x = 2 -> 1`. Nếu công thức của bạn cho `x = 1 -> 1` hay `x = 2 -> 2` thì sai ngay.

**Mẹo test ngẫu nhiên:**

- Đưa nguồn ngẫu nhiên thành thứ có thể *thay thế* (tham số `rng`, hoặc patch `random.randint`) để chạy cố định và liệt kê được mọi khả năng.
- Luôn tự hỏi "mỗi kết quả tương ứng bao nhiêu chuỗi lựa chọn?" — đó chính là xác suất.
- Seed cố định → test lặp lại được; nhưng đừng quên test "code sai phải bị bắt".

**Hay gặp cùng pattern:** LeetCode 497 (điểm ngẫu nhiên trong các hình chữ nhật không giao nhau, trọng số = diện tích), 710 (chọn ngẫu nhiên ngoài blacklist), 398 (chọn ngẫu nhiên chỉ số bằng target — reservoir sampling), 384 (xáo mảng).

## 15. Biến thể

1. **Cập nhật trọng số** `update(i, new_w)` xen kẽ `pickIndex`: mảng tiền tố phải dựng lại mỗi lần cập nhật (`O(n)`). Dùng **cây Fenwick (BIT)** để cả hai thao tác đều `O(log n)` — bài tập 5.
2. **Chọn không hoàn lại**: chọn xong, xoá chỉ số đó khỏi tập. Cũng dùng Fenwick: sau khi chọn, `update(i, 0)` và tổng giảm.
3. **Điểm ngẫu nhiên trong các hình chữ nhật** (LeetCode 497): trọng số = số điểm nguyên của mỗi hình, chọn hình theo cách của bài này rồi chọn điểm đều trong hình.
4. **Trọng số thực**: dùng `random.random() * S` và `bisect_right`; chú ý sai số.
5. **Trọng số có thể bằng 0**: tiền tố không tăng chặt; `bisect_left` vẫn đúng (luôn trả chỉ số đầu tiên đạt ngưỡng, bỏ qua các chỉ số trọng số 0), nhưng alias và vài cách khác cần cẩn thận.
6. **Thư viện sẵn có:** `random.choices(population, weights=w)` chính là cách này, đáng biết để kiểm chứng.

In [35]:
# Biến thể 5: trọng số có thể bằng 0 -> bisect_left vẫn đúng và không bao giờ chọn chỉ số trọng số 0
class PickWithZeros:
    def __init__(self, w):
        self.prefix = list(accumulate(w))
    def pickIndex(self):
        return bisect.bisect_left(self.prefix, random.randint(1, self.prefix[-1]))


w0 = [0, 2, 0, 0, 3, 0]
dist = exact_picks(PickWithZeros, w0)
print({k: str(v) for k, v in dist.items()})
assert dist == {1: Fraction(2, 5), 4: Fraction(3, 5)}

# Biến thể 6: random.choices làm đối chứng
random.seed(3)
cnt = Counter(random.choices(range(4), weights=[3, 1, 2, 4], k=100000))
stat, p = chi_square_test([cnt[i] for i in range(4)], [3, 1, 2, 4])
print("random.choices:", dict(sorted(cnt.items())), f"p={p:.3f}")

{1: '2/5', 4: '3/5'}
random.choices: {0: 30120, 1: 10011, 2: 20003, 3: 39866} p=0.815


## 16. So sánh các cách

| Cách | Ý tưởng | Time (init / pick) | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Nhân bản | Lập bảng `S` ô → chỉ số rồi chọn đều | `O(S)` / `O(1)` | `O(S)` | Chỉ để hiểu đề với `S` rất nhỏ; không nộp được |
| 2. Prefix + quét | Cộng dồn, tìm đoạn chứa `x` bằng duyệt | `O(n)` / `O(n)` | `O(n)` | `n` nhỏ, ít lần gọi; code ngắn nhất |
| 3. Prefix + nhị phân | Cộng dồn, tìm kiếm nhị phân `x` | `O(n)` / `O(log n)` | `O(n)` | **Đáp án chuẩn**, phỏng vấn và nộp bài |
| 4. Alias (Vose) | `n` cột sức chứa bằng nhau, mỗi cột tối đa 2 chỉ số | `O(n)` / `O(1)` | `O(n)` | Pick cực nhiều lần, trọng số cố định; bù bằng code phức tạp và tốn 2 số ngẫu nhiên |

Điểm chung: cả 4 cách cho **phân phối đúng tuyệt đối** (đã chứng minh bằng đếm `Fraction` ở mục 10 và 12). Khác nhau chỉ ở tốc độ và bộ nhớ. Nếu có `update` xen kẽ, nên chuyển sang Fenwick (bài tập 5).

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell TODO bên dưới **chưa được gọi** — bạn tự điền, rồi bỏ comment dòng test ở cuối mỗi cell.

### Bài 1 (đoán output)

Đoán trước kết quả in ra rồi mới chạy:

In [36]:
w = [2, 5, 3]
p = list(accumulate(w))
print(p)
print(bisect.bisect_left(p, 1), bisect.bisect_left(p, 2), bisect.bisect_left(p, 3),
      bisect.bisect_left(p, 7), bisect.bisect_left(p, 8), bisect.bisect_left(p, 10))

[2, 7, 10]
0 0 1 1 2 2


### Bài 2 (tính tay)

`w = [4, 1, 1, 2]`.

- (a) Tổng tiền tố là gì và `S` bằng bao nhiêu?
- (b) Với `x = 6`, `pickIndex` trả chỉ số nào? Vì sao?
- (c) Xác suất chọn chỉ số `3` là bao nhiêu? Có bao nhiêu giá trị `x` ứng với chỉ số `2`?

Viết lời giải ra giấy rồi kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện hàm tìm kiếm nhị phân định vị. Chỉ số cần tìm là chỉ số nhỏ nhất có `prefix[i] >= x`:

In [37]:
def locate_todo(prefix, x):
    lo, hi = 0, len(prefix) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        # TODO: nếu prefix[mid] >= x thì ... ; ngược lại thì ...
        pass
    # TODO: trả về đáp án
    return None


# Khi làm xong, bỏ comment các dòng sau để tự kiểm tra:
# assert [locate_todo([3, 4, 6, 10], x) for x in range(1, 11)] == [0, 0, 0, 1, 2, 2, 3, 3, 3, 3]

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết lớp `MySolution` (cách chuẩn: tiền tố + nhị phân hoặc `bisect`). Gợi ý khung:

In [38]:
class MySolution:
    def __init__(self, w):
        pass            # TODO

    def pickIndex(self):
        pass            # TODO


# Khi làm xong, kiểm tra tuyệt đối:
# assert exact_picks(MySolution, [3, 1, 2, 4]) == expected([3, 1, 2, 4])

### Bài 5 (mini challenge): trọng số thay đổi

Thêm method `update(i, new_w)` để đổi trọng số `w[i]` thành `new_w >= 1`. Cả `update` và `pickIndex` đều phải `O(log n)`. Gợi ý: **cây Fenwick** lưu tổng theo khoảng và hỗ trợ "tìm chỉ số nhỏ nhất có tổng tiền tố `>= x`" bằng cách *đi xuống theo luỹ thừa của 2*.

In [39]:
class DynamicPicker:
    def __init__(self, w):
        pass            # TODO: dựng Fenwick

    def update(self, i, new_w):
        pass            # TODO

    def pickIndex(self):
        pass            # TODO


# Khi làm xong:
# d = DynamicPicker([1, 3]); d.update(0, 3)
# assert exact_distribution(d.pickIndex) == {0: Fraction(1, 2), 1: Fraction(1, 2)}

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: `accumulate` làm gì với `[2, 5, 3]`?
- Mức 2: `bisect_left(p, x)` trả chỉ số nhỏ nhất có `p[i] >= x`.
- Mức 3: `p = [2, 7, 10]`. Hãy tìm với từng `x`.

**Bài 2**

- Mức 1: vẽ thanh gỗ dài `S` và chia 4 đoạn.
- Mức 2: chỉ số `i` sở hữu các ô `prefix[i-1] + 1 ... prefix[i]`.
- Mức 3: `prefix = [4, 5, 6, 8]`; ô `6` nằm ở chỗ nào?

**Bài 3**

- Mức 1: cửa sổ `[lo, hi]` luôn chứa đáp án.
- Mức 2: nếu `prefix[mid] >= x` thì `mid` còn có thể là đáp án, không được bỏ nó.
- Mức 3: `hi = mid` khi `>=`; `lo = mid + 1` khi `<`; cuối cùng `return lo`.

**Bài 4**

- Mức 1: việc nào nên làm ở `__init__`?
- Mức 2: lưu `prefix` (và có thể cả `S = prefix[-1]`).
- Mức 3: `x = random.randint(1, S)`; rồi định vị `x` như bài 3.

**Bài 5**

- Mức 1: Fenwick có hai thao tác: cộng `delta` vào vị trí và hỏi tổng tiền tố.
- Mức 2: `update(i, new_w)` = cộng `new_w - w[i]` (nhớ lưu `w` hiện tại).
- Mức 3: tìm chỉ số theo `x`: `pos = 0`; với `pw` từ lớn nhất (luỹ thừa 2 `<= n`) xuống 1: nếu `pos + pw <= n` và `tree[pos + pw] < x` thì `pos += pw`, `x -= tree[pos]`. Đáp án là `pos` (0-indexed).

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert` để chạy kiểm tra.

### Bài 1

In [40]:
p = list(accumulate([2, 5, 3]))
assert p == [2, 7, 10]
res = [bisect.bisect_left(p, x) for x in (1, 2, 3, 7, 8, 10)]
assert res == [0, 0, 1, 1, 2, 2]
print("Bài 1:", p, res)

Bài 1: [2, 7, 10] [0, 0, 1, 1, 2, 2]


### Bài 2

(a) `prefix = [4, 5, 6, 8]`, `S = 8`. (b) `x = 6`: `prefix[2] = 6 >= 6` và `prefix[1] = 5 < 6`, nên chỉ số `2`. (c) Xác suất chỉ số `3` là `2/8 = 1/4`; chỉ số `2` sở hữu đúng `w[2] = 1` giá trị `x` (đó là `x = 6`).

In [41]:
w = [4, 1, 1, 2]
prefix = list(accumulate(w))
assert prefix == [4, 5, 6, 8] and prefix[-1] == 8
assert bisect.bisect_left(prefix, 6) == 2
owners = [bisect.bisect_left(prefix, x) for x in range(1, 9)]
assert owners.count(2) == 1 and owners.index(2) + 1 == 6
assert Fraction(owners.count(3), 8) == Fraction(1, 4)
print("Bài 2:", owners)

Bài 2: [0, 0, 0, 0, 1, 2, 3, 3]


### Bài 3

In [42]:
def locate_answer(prefix, x):
    lo, hi = 0, len(prefix) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if prefix[mid] >= x:
            hi = mid
        else:
            lo = mid + 1
    return lo


assert [locate_answer([3, 4, 6, 10], x) for x in range(1, 11)] == [0, 0, 0, 1, 2, 2, 3, 3, 3, 3]
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [43]:
class MySolutionAnswer:
    def __init__(self, w):
        self.prefix = list(accumulate(w))

    def pickIndex(self):
        return bisect.bisect_left(self.prefix, random.randint(1, self.prefix[-1]))


assert exact_picks(MySolutionAnswer, [3, 1, 2, 4]) == expected([3, 1, 2, 4])
print("Bài 4 OK")

Bài 4 OK


### Bài 5 (Fenwick)

`tree[k]` (1-indexed) lưu tổng của một đoạn kết thúc tại `k`; độ dài đoạn là `k & -k`.

In [44]:
class DynamicPickerAnswer:
    def __init__(self, w):
        self.n = len(w)
        self.w = list(w)
        self.tree = [0] * (self.n + 1)
        for i, x in enumerate(w):
            self._add(i + 1, x)
        self.top = 1
        while self.top * 2 <= self.n:
            self.top *= 2

    def _add(self, k, delta):
        while k <= self.n:
            self.tree[k] += delta
            k += k & -k

    def total(self):
        k, s = self.n, 0
        while k:
            s += self.tree[k]
            k -= k & -k
        return s

    def update(self, i, new_w):
        self._add(i + 1, new_w - self.w[i])
        self.w[i] = new_w

    def pickIndex(self):
        x = random.randint(1, self.total())
        pos, pw = 0, self.top
        while pw:
            nxt = pos + pw
            if nxt <= self.n and self.tree[nxt] < x:
                pos = nxt
                x -= self.tree[nxt]
            pw >>= 1
        return pos                      # 0-indexed


d = DynamicPickerAnswer([1, 3])
assert exact_distribution(d.pickIndex) == {0: Fraction(1, 4), 1: Fraction(3, 4)}
d.update(0, 3)
assert exact_distribution(d.pickIndex) == {0: Fraction(1, 2), 1: Fraction(1, 2)}
d.update(1, 1)
assert exact_distribution(d.pickIndex) == {0: Fraction(3, 4), 1: Fraction(1, 4)}

# đối chiếu ngẫu nhiên với tiền tố + bisect sau mỗi lần cập nhật
rng = XorShift64(555)
w = [rng.randint(1, 6) for _ in range(9)]
d = DynamicPickerAnswer(w)
for _ in range(40):
    i, v = rng.randrange(len(w)), rng.randint(1, 6)
    w[i] = v
    d.update(i, v)
    assert exact_distribution(d.pickIndex) == expected(w)
print("Bài 5 OK: Fenwick đúng sau 40 lần cập nhật ngẫu nhiên (đếm chính xác)")

Bài 5 OK: Fenwick đúng sau 40 lần cập nhật ngẫu nhiên (đếm chính xác)


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (xác suất theo trọng số):**

```python
class Solution:
    def __init__(self, w):
        self.prefix = list(accumulate(w))          # làm 1 lần

    def pickIndex(self):
        x = random.randint(1, self.prefix[-1])      # chọn đều ô trong [1, S]
        lo, hi = 0, len(self.prefix) - 1
        while lo < hi:                               # chỉ số nhỏ nhất có prefix[i] >= x
            mid = (lo + hi) // 2
            if self.prefix[mid] >= x:
                hi = mid
            else:
                lo = mid + 1
        return lo
```

**Checklist trước khi submit:**

- [ ] Tiền tố dựng **một lần** trong `__init__`, không dựng lại trong `pickIndex`.
- [ ] `randint(1, S)` (hai đầu đều lấy) đi cùng `bisect_left` / `prefix[mid] >= x`; hoặc `randrange(S)` đi cùng `bisect_right`.
- [ ] Mảng tiền tố cuối cùng bằng `S`, và `x = 1` → chỉ số `0`, `x = S` → chỉ số `n - 1`.
- [ ] `n = 1` luôn trả `0`; không nhân bản `w[i]` bản sao (`S` tới `10^9`).
- [ ] Đã test **đếm chính xác** (liệt kê mọi `x`) hoặc chi-bình-phương có seed, không so đáp án cứng.
- [ ] Không in debug; tên class, method, tham số đúng skeleton LeetCode.